In [1]:
!pip install -q mediapipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.9/37.9 MB 46.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.4/137.4 kB 11.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.


In [2]:
!wget -q https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task

In [3]:
import os, kagglehub, json
import cv2
import numpy as np
import pandas as pd
import mediapipe as mp
from tqdm import tqdm
import tensorflow as tf

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# MediaPipe Hand Landmarker
BaseOptions = python.BaseOptions
HandLandmarker = vision.HandLandmarker
HandLandmarkerOptions = vision.HandLandmarkerOptions
RunningMode = vision.RunningMode


In [4]:
BASE_PATH = '/kaggle/input/datasets/risangbaskoro/wlasl-processed'

print(os.listdir('/kaggle/input'))
print(os.listdir(BASE_PATH))

['datasets']
['nslt_2000.json', 'videos', 'nslt_1000.json', 'WLASL_v0.3.json', 'wlasl_class_list.txt', 'nslt_300.json', 'missing.txt', 'nslt_100.json']


In [5]:
with open(f'{BASE_PATH}/nslt_100.json', 'r') as f:
    data_100 = json.load(f)

print(f"Type: {type(data_100)}")
print(f"Number of entries: {len(data_100)}")

for i, (key, value) in enumerate(data_100.items()):
    print(f"\nEntry {i+1}:")
    print(f"  Key: {key}")
    print(f"  Value: {value}")
    if i == 2:
        break

Type: <class 'dict'>
Number of entries: 2038

Entry 1:
  Key: 05237
  Value: {'subset': 'train', 'action': [77, 1, 55]}

Entry 2:
  Key: 69422
  Value: {'subset': 'val', 'action': [27, 1, 51]}

Entry 3:
  Key: 10899
  Value: {'subset': 'train', 'action': [82, 1, 48]}


In [6]:

with open(f'{BASE_PATH}/WLASL_v0.3.json', 'r') as f:
    wlasl_data = json.load(f)

print(f"Type: {type(wlasl_data)}")
print(f"Number of entries: {len(wlasl_data)}")

# Show first entry structure
print(f"\nFirst entry:")
print(json.dumps(wlasl_data[0], indent=2))



Type: <class 'list'>
Number of entries: 2000

First entry:
{
  "gloss": "book",
  "instances": [
    {
      "bbox": [
        385,
        37,
        885,
        720
      ],
      "fps": 25,
      "frame_end": -1,
      "frame_start": 1,
      "instance_id": 0,
      "signer_id": 118,
      "source": "aslbrick",
      "split": "train",
      "url": "http://aslbricks.org/New/ASL-Videos/book.mp4",
      "variation_id": 0,
      "video_id": "69241"
    },
    {
      "bbox": [
        190,
        25,
        489,
        370
      ],
      "fps": 25,
      "frame_end": -1,
      "frame_start": 1,
      "instance_id": 1,
      "signer_id": 90,
      "source": "aslsignbank",
      "split": "train",
      "url": "https://aslsignbank.haskins.yale.edu/dictionary/protected_media/glossvideo/ASL/BO/BOOK-418.mp4",
      "variation_id": 0,
      "video_id": "65225"
    },
    {
      "bbox": [
        262,
        1,
        652,
        480
      ],
      "fps": 25,
      "frame_end": -1,
   

In [7]:
BASE_PATH = '/kaggle/input/datasets/risangbaskoro/wlasl-processed'
VIDEO_PATH = f'{BASE_PATH}/videos'
OUT_PATH  = '/kaggle/working/'
SEQUENCE_LENGTH = 30   # frames per video
NUM_LANDMARKS   = 63   # 21 hand landmarks × 3 (x, y, z)
NUM_HANDS = 2
TOTAL_FEATURES = NUM_HANDS * NUM_LANDMARKS
HAND_TASK_PATH = "/kaggle/working/hand_landmarker.task"

In [8]:
with open(f'{BASE_PATH}/WLASL_v0.3.json', 'r') as f:
    wlasl_data = json.load(f)
    
with open(f'{BASE_PATH}/nslt_100.json', 'r') as f:
    nslt_100 = json.load(f)

with open(f'{BASE_PATH}/missing.txt', 'r') as f:
    missing = set(f.read().splitlines())

In [9]:
# video_id to word
video_metadata = {}
for entry in wlasl_data:
    word = entry['gloss'] # gloss is the word
    for instance in entry['instances']: # instances is specific video example of that gloss
        video_id = instance['video_id']
        video_metadata[video_id] = {
            "word": word,
            "bbox": instance.get("bbox"),
            "fps": instance.get("fps", 25),
            "frame_start": instance.get("frame_start", 1),
            "frame_end": instance.get("frame_end", -1),
            "signer_id": instance.get("signer_id"),
            "source": instance.get("source"),
        }

In [10]:
# Filter to only 100 word subset and available videos
dataset = []
for video_id, info in nslt_100.items():
    video_file = f"{video_id}.mp4"
    video_path = os.path.join(VIDEO_PATH, video_file)
    
    # Skip missing or unavailable videos
    if video_id in missing:
        continue
        
    # Skip files that don't exist
    if not os.path.exists(video_path):
        continue
        
    # Skip videos without metadata
    if video_id not in video_metadata:
        continue

    metadata = video_metadata[video_id]
    
    dataset.append({
        "video_id": video_id,
        "video_path": video_path,
        "word": metadata["word"],
        "subset": info["subset"],

        # useful video metadata
        "bbox": metadata["bbox"],
        "fps": metadata["fps"],
        "frame_start": metadata["frame_start"],
        "frame_end": metadata["frame_end"],
        "signer_id": metadata["signer_id"],
        "source": metadata["source"],
    })

In [11]:
# Get unique words
words = sorted(list(set([d['word'] for d in dataset])))
word2idx = {word: idx for idx, word in enumerate(words)}
idx2word = {idx: word for idx, word in enumerate(words)}
NUM_CLASSES = len(words)

print(f"Total usable videos : {len(dataset)}")
print(f"Number of classes : {NUM_CLASSES}")
print(f"Words: {words}")

Total usable videos : 1013
Number of classes : 100
Words: ['accident', 'africa', 'all', 'apple', 'basketball', 'bed', 'before', 'bird', 'birthday', 'black', 'blue', 'book', 'bowling', 'brown', 'but', 'can', 'candy', 'chair', 'change', 'cheat', 'city', 'clothes', 'color', 'computer', 'cook', 'cool', 'corn', 'cousin', 'cow', 'dance', 'dark', 'deaf', 'decide', 'doctor', 'dog', 'drink', 'eat', 'enjoy', 'family', 'fine', 'finish', 'fish', 'forget', 'full', 'give', 'go', 'graduate', 'hat', 'hearing', 'help', 'hot', 'how', 'jacket', 'kiss', 'language', 'last', 'later', 'letter', 'like', 'man', 'many', 'medicine', 'meet', 'mother', 'need', 'no', 'now', 'orange', 'paint', 'paper', 'pink', 'pizza', 'play', 'pull', 'purple', 'right', 'same', 'school', 'secretary', 'shirt', 'short', 'son', 'study', 'table', 'tall', 'tell', 'thanksgiving', 'thin', 'thursday', 'time', 'walk', 'want', 'what', 'white', 'who', 'woman', 'work', 'wrong', 'year', 'yes']


In [12]:
print(dataset[0])
print(dataset[10])

print("Train:", sum(d["subset"] == "train" for d in dataset))
print("Val  :", sum(d["subset"] == "val" for d in dataset))
print("Test :", sum(d["subset"] == "test" for d in dataset))

{'video_id': '69422', 'video_path': '/kaggle/input/datasets/risangbaskoro/wlasl-processed/videos/69422.mp4', 'word': 'orange', 'subset': 'val', 'bbox': [363, 40, 871, 720], 'fps': 25, 'frame_start': 1, 'frame_end': -1, 'signer_id': 118, 'source': 'aslbrick'}
{'video_id': '51061', 'video_path': '/kaggle/input/datasets/risangbaskoro/wlasl-processed/videos/51061.mp4', 'word': 'shirt', 'subset': 'train', 'bbox': [53, 0, 527, 480], 'fps': 25, 'frame_start': 1, 'frame_end': -1, 'signer_id': 13, 'source': 'asldeafined'}
Train: 748
Val  : 165
Test : 100


In [13]:
options = HandLandmarkerOptions(
    base_options=BaseOptions(
        model_asset_path=str(HAND_TASK_PATH) # the downloaded task
    ),
    
    running_mode=RunningMode.VIDEO,
    
    num_hands=2, # We want both hands
    
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

landmarker = HandLandmarker.create_from_options(options)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1788377899.100210     152 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788377899.122159     152 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [14]:
def normalize_hand(hand_landmarks):
    # 21 points × (x, y, z)
    coords = np.array(
        [[landmark.x, landmark.y, landmark.z] for landmark in hand_landmarks], dtype=np.float32
    )

    # Make the wrist at the origin
    wrist = coords[0].copy()
    coords = coords - wrist

    # Scaling
    scale = np.linalg.norm(coords[9])

    if scale < 1e-6:
        return None

    coords = coords / scale

    return coords.flatten()

In [19]:
def extract_frame_features(result):
    # Default: no hand detected
    left_hand = np.zeros(NUM_LANDMARKS, dtype=np.float32)
    right_hand = np.zeros(NUM_LANDMARKS, dtype=np.float32)
    
    if not result.hand_landmarks:
        return np.concatenate([left_hand, right_hand])

    for hand_landmarks, handedness in zip(result.hand_landmarks, result.handedness):

        normalized = normalize_hand(hand_landmarks)

        if normalized is None:
            continue

        # MediaPipe handedness
        hand_label = handedness[0].category_name.lower()

        if hand_label == "left":
            left_hand = normalized
            
        elif hand_label == "right":
            right_hand = normalized
            
    # Final frame representation
    features = np.concatenate([left_hand, right_hand])

    return features

extraction action

In [23]:
def extract_video_landmarks(video_path, options, frame_start=1, frame_end=-1):
    landmarker = HandLandmarker.create_from_options(options)
    cap = cv2.VideoCapture(video_path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    features = []
    frame_idx = 0

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame_idx += 1
        if frame_idx < frame_start:
            continue
        if frame_end != -1 and frame_idx > frame_end:
            break

        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        timestamp_ms = int(frame_idx * (1000 / fps))

        result = landmarker.detect_for_video(mp_image, timestamp_ms)
        features.append(extract_frame_features(result))

    cap.release()
    landmarker.close()
    return np.array(features, dtype=np.float32)

In [24]:
def sample_sequence(features, sequence_length=SEQUENCE_LENGTH):
    n_frames = len(features)
    if n_frames == 0:
        return None  # Signal that extraction failed

    # Evenly sample or stretch indices across the entire sequence length
    idx = np.linspace(0, n_frames - 1, sequence_length).astype(int)
    return features[idx]

In [26]:
## DO NOT RUN THIS CELLLLLLLLLLLLLLLLLLLLLLLLLL!!!!!!!!!!!!!!!!!!!!!
# takes toooooooo much time ~30 mins!!
# idk the problem im looking into it
X, y, subsets = [], [], []

for d in tqdm(dataset):
    raw = extract_video_landmarks(
        d["video_path"], options,
        frame_start=d["frame_start"], frame_end=d["frame_end"]
    )
    seq = sample_sequence(raw)
    X.append(seq)
    y.append(word2idx[d["word"]])
    subsets.append(d["subset"])

X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int32)
subsets = np.array(subsets)

np.save(f"{OUT_PATH}/X.npy", X)
np.save(f"{OUT_PATH}/y.npy", y)
np.save(f"{OUT_PATH}/subsets.npy", subsets)

X_train, y_train = X[subsets == "train"], y[subsets == "train"]
X_val,   y_val   = X[subsets == "val"],   y[subsets == "val"]
X_test,  y_test  = X[subsets == "test"],  y[subsets == "test"]

  0%|          | 0/1013 [00:00<?, ?it/s]W0000 00:00:1788378387.463078     202 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788378387.484160     201 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  0%|          | 1/1013 [00:02<36:11,  2.15s/it]W0000 00:00:1788378389.593198     214 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1788378389.611365     214 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
  0%|          | 2/1013 [00:03<27:03,  1.61s/it]W0000 00:00:1788378390.826707     228 inference_feedback_manager.cc:121] Feedback manager requires a model with a single signature infer

In [27]:
  X = np.load(f"{OUT_PATH}/X.npy")
  y = np.load(f"{OUT_PATH}/y.npy")
  subsets = np.load(f"{OUT_PATH}/subsets.npy")

In [29]:
model = tf.keras.Sequential([
    tf.keras.layers.Masking(mask_value=0.0, input_shape=(SEQUENCE_LENGTH, TOTAL_FEATURES)),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(128, return_sequences=True, use_cudnn=False)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, use_cudnn=False)),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dense(NUM_CLASSES, activation="softmax"),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=12, restore_best_weights=True, monitor="val_accuracy"),
    tf.keras.callbacks.ReduceLROnPlateau(patience=5, factor=0.5, monitor="val_loss"),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=callbacks,
)

model.evaluate(X_test, y_test)

Epoch 1/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 18s 207ms/step - accuracy: 0.0281 - loss: 4.5414 - val_accuracy: 0.0545 - val_loss: 4.3839 - learning_rate: 0.0010
Epoch 2/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 189ms/step - accuracy: 0.0856 - loss: 4.1256 - val_accuracy: 0.0970 - val_loss: 3.9716 - learning_rate: 0.0010
Epoch 3/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 188ms/step - accuracy: 0.1604 - loss: 3.6110 - val_accuracy: 0.1515 - val_loss: 3.6723 - learning_rate: 0.0010
Epoch 4/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 188ms/step - accuracy: 0.2099 - loss: 3.2030 - val_accuracy: 0.1636 - val_loss: 3.3648 - learning_rate: 0.0010
Epoch 5/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 187ms/step - accuracy: 0.2380 - loss: 2.9104 - val_accuracy: 0.2364 - val_loss: 3.1453 - learning_rate: 0.0010
Epoch 6/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 189ms/step - accuracy: 0.2928 - loss: 2.6821 - val_accuracy: 0.1879 - val_loss: 3.0811 - learning_rate: 0.0010
Epoch 7/100
47/47 ━━━━━━━━━━━━━━━━━━━━ 9s 190ms/step - accuracy: 0.3302 - loss: 2

[2.4506309032440186, 0.5299999713897705]

In [31]:
model.save(f"{OUT_PATH}/wlasl_100_model_1.keras")
model.save(f"{OUT_PATH}/wlasl_100_model_1.h5")
import json
with open(f"{OUT_PATH}/idx2word.json", "w") as f:
    json.dump(idx2word, f)